In [ ]:
from pathlib import Path

test_bids_write_path = Path("/tmp/bids_test")

# Writing a new BIDS dataset

In [ ]:
import json
import subprocess
from shutil import rmtree

from clinicaio import (
    BIDSDataset,
    BIDSDatasetDescription,
    BIDSDatasetType,
    DataType,
    FileExtension,
    ImageScanInfo,
    SessionInfo,
    SubjectInfo,
)

rmtree(test_bids_write_path, ignore_errors=True)

dataset = BIDSDataset(
    bids_path=test_bids_write_path,
    description=BIDSDatasetDescription.new(
        BIDSDatasetType.RAW, name="TEST BIDS1", bids_version="1.10.0"
    ),
)
subject = dataset.add_subject(
    id="sub-ONE",
    info=SubjectInfo(
        {
            "data1": 345,
            "data2": "texte",
            "data3": None,
        }
    ),
)

session = subject.add_session(
    id="ses-ONE",
    info=SessionInfo(
        acquisition_time="2026-04-22T01:02:03Z",
        foo=37,
        bar=3929,
        baz="texte",
    ),
)
session2 = subject.add_session(
    id="ses-N2",
    info=SessionInfo(
        acquisition_time="2026-04-20T11:12:13Z",
        foo=37,
        bar=3929,
        baz="texte",
    ),
)

image1 = session.write_image(
    data_type=DataType.PET,
    nifti_extension=FileExtension.NII_GZ,
    entities={"trc": "18FFDG", "task": "rest"},
    suffix="T1w",
    scan_info=ImageScanInfo(
        a="1a",
        b="1b",
    ),
)
with open(image1.get_nifti_image_path(), mode="x") as f:
    print("NIFTI1", file=f)

image2 = session.write_image(
    data_type=DataType.ANAT,
    nifti_extension=FileExtension.NII_GZ,
    entities={"trc": "11CPIB", "task": "rest"},
    suffix="T1w",
    scan_info=ImageScanInfo(
        b="2b",
        c="2c",
    ),
)
with open(image2.get_nifti_image_path(), mode="x") as f:
    print("NIFTI2", file=f)

with open(image1.get_image_companion_path(FileExtension.JSON), "x") as f:
    json.dump(
        obj={
            "name": "test",
            "size": 3092,
        },
        fp=f,
    )

dataset.write_to_folder(readme="TEST README BIDS")
with dataset.write_root_file("LICENSE") as f:
    print("foobar", file=f)
with dataset.write_root_file("data.bin", write_binary=True) as f:
    f.write(b"foobar")
    f.flush()

## Show the written BIDS dataset

In [ ]:
import pandas as pd

read_infos = True
dataset2 = BIDSDataset.populate_from_dir(
    test_bids_write_path,
    sessions_info=read_infos,
    subjects_info=read_infos,
    image_scans_info=read_infos,
)
# pprint(dataset2)
print(dataset2.description)
for subject in dataset2.all_subjects():
    print(subject.id, subject.info)
    for session in subject.all_sessions():
        print("\t", session.id, session.info)
        for image in session.all_images():
            print(f"\t\t{image.data_type}: {image}")

subprocess.run(["find", "."], check=True, cwd=test_bids_write_path)

pd.set_option("display.max_colwidth", None)
pd.read_csv(
    test_bids_write_path / "sub-ONE" / "ses-ONE" / "sub-ONE_ses-ONE_scans.tsv",
    sep="\t",
    dtype=str,
    keep_default_na=False,
)